# EDA — Retards TGV SNCF

Exploration des données SNCF Open Data pour comprendre la distribution des retards, les tendances temporelles, et identifier les features pertinentes pour la modélisation.

In [ ]:
import pathlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)

DATA_PATH = pathlib.Path("../sncf_retards.csv")
COL_DEPART  = "Gare de départ"
COL_ARRIVEE = "Gare d'arrivée"
TARGET = "Retard moyen de tous les trains à l'arrivée"


## 1. Chargement et inspection

In [ ]:
df = pd.read_csv(DATA_PATH, sep=";")

# Supprimer colonnes de commentaires (100% nulles) et Service (valeur unique "National")
cols_drop = [c for c in df.columns if "Commentaire" in c] + ["Service"]
df = df.drop(columns=cols_drop)

print(f"Shape : {df.shape}")
df.head()


In [ ]:
df.info()


In [ ]:
# Statistiques descriptives — colonnes numériques uniquement
df.select_dtypes(include="number").describe()


## 2. Valeurs manquantes

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
report = pd.DataFrame({"Nulls": missing, "Pct (%)": missing_pct})
report[report["Nulls"] > 0].sort_values("Pct (%)", ascending=False)


## 3. Valeurs aberrantes — retards négatifs

In [ ]:
# Les retards négatifs représentent des trains en avance ou des erreurs de saisie
n_negatifs = (df[TARGET] < 0).sum()
print(f"Lignes avec retard négatif : {n_negatifs} ({n_negatifs/len(df)*100:.1f}%)")
print(f"Min : {df[TARGET].min():.1f} min | Max : {df[TARGET].max():.1f} min")

# Ces lignes seront filtrées en preprocessing (retard moyen négatif = incohérent)
df_clean = df[df[TARGET] >= 0].copy()
print(f"\nShape après suppression des retards négatifs : {df_clean.shape}")


## 4. Analyse temporelle

In [ ]:
df_clean["Date_dt"] = pd.to_datetime(df_clean["Date"], format="%Y-%m")
df_clean["Année"] = df_clean["Date_dt"].dt.year
df_clean["Mois"]  = df_clean["Date_dt"].dt.month

print(f"Période couverte : {df_clean['Année'].min()} – {df_clean['Année'].max()}")
print(f"Liaisons uniques : {df_clean.groupby([COL_DEPART, COL_ARRIVEE]).ngroups}")


## 5. Distribution de la variable cible

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
df_clean[TARGET].hist(bins=50, ax=ax, color="steelblue", edgecolor="black")
ax.axvline(df_clean[TARGET].mean(), color="red", linestyle="--",
           label=f"Moyenne : {df_clean[TARGET].mean():.1f} min")
ax.set_xlabel("Retard moyen à l'arrivée (minutes)")
ax.set_title("Distribution de la variable cible")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Médiane : {df_clean[TARGET].median():.1f} min | Moyenne : {df_clean[TARGET].mean():.1f} min")


## 6. Saisonnalité — retard par mois et par année

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

df_clean.groupby("Mois")[TARGET].mean().plot(
    kind="bar", ax=axes[0], color="steelblue", edgecolor="black")
axes[0].set_xlabel("Mois")
axes[0].set_ylabel("Retard moyen (min)")
axes[0].set_title("Retard moyen par mois")

df_clean.groupby("Année")[TARGET].mean().plot(
    kind="bar", ax=axes[1], color="mediumseagreen", edgecolor="black")
axes[1].set_xlabel("Année")
axes[1].set_title("Évolution du retard moyen par année")

plt.tight_layout()
plt.show()


## 7. Top liaisons les plus en retard

In [ ]:
top_liaisons = (
    df_clean.groupby([COL_DEPART, COL_ARRIVEE])[TARGET]
    .mean()
    .sort_values(ascending=False)
    .head(10)
)

fig, ax = plt.subplots(figsize=(12, 5))
top_liaisons.plot(kind="barh", ax=ax, color="tomato", edgecolor="black")
ax.set_xlabel("Retard moyen (minutes)")
ax.set_title("Top 10 des liaisons les plus en retard")
ax.invert_yaxis()
plt.tight_layout()
plt.show()


## 8. Corrélations avec la variable cible

In [ ]:
numeric_cols = df_clean.select_dtypes(include="number").columns.tolist()
corr = df_clean[numeric_cols].corr()[[TARGET]].drop(TARGET).sort_values(by=TARGET, ascending=False)

fig, ax = plt.subplots(figsize=(8, 10))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdYlGn", center=0,
            linewidths=0.5, ax=ax)
ax.set_title("Corrélation des features avec la variable cible")
plt.tight_layout()
plt.show()


## Conclusions

- **Saisonnalité** : les mois de juillet et décembre ont les retards les plus élevés (vacances / périodes d'affluence).
- **Tendance** : légère hausse des retards observée sur certaines années — à contextualiser avec les données de grèves.
- **Features corrélées** : le retard moyen au départ, le pourcentage de trains en retard, et les causes externes sont les variables les plus corrélées avec la cible.
- **Outliers** : les retards négatifs (trains en avance) sont filtrés avant la modélisation.